# non_gen_zero_shot_ablations

This notebook runs **phase-1** non-generative zero-shot ablations by scoring each candidate answer and selecting the highest-probability option.

## Modeling Pipeline

- **Adaptation layer**: frozen pretrained SmolVLM (no fine-tuning).
- **Decision layer**: score each answer candidate under prompt context.
- **Phase-1 scope**: subset ablation screening only (no phase-2/3).
- **Outputs**: validation tables, optional submission, and save-gated artifacts.

In [1]:
from pathlib import Path
from itertools import product
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

import torch
import transformers
from transformers import AutoProcessor


## Colab + Drive Setup

This setup supports local Jupyter and Google Colab. In Colab, Drive is mounted to access this repo.

In [2]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    CANDIDATE_ROOT = Path('/content/drive/MyDrive/DL_Final_Project')
    PROJECT_ROOT = CANDIDATE_ROOT if CANDIDATE_ROOT.exists() else Path.cwd()
else:
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        PROJECT_ROOT = cwd.parent
    elif cwd.name == 'zero_shot_study' and cwd.parent.name == 'notebooks':
        PROJECT_ROOT = cwd.parent.parent
    else:
        PROJECT_ROOT = cwd

print('IN_COLAB:', IN_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.zero_shot_ablation_utils import (
    set_seed,
    load_split,
    apply_sanity_subset,
    cap_validation_rows,
    select_ablation_configs,
    build_block_configs,
    run_nongenerative_ablation,
    summarize_nongenerative_predictions,
    top_configs,
    make_submission,
    validate_submission,
    build_failure_examples,
    maybe_save_dataframe,
    maybe_save_json,
    maybe_save_figure,
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
IN_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/DL_Final_Project


## Configuration

Use `selected_ablation_ids` to run only specific ablations. Use `max_val_examples` to cap validation rows used in evaluation.

Set `batch_size` based on available GPU memory. Start small (for example, 1-4) and increase gradually.

In [3]:
save = True
generate_submission = False
sanity_check = True
n = 512
seed = 42

max_val_examples = 128
top_k = 5
batch_size = 64
load_saved_results = True

DATA_DIR = PROJECT_ROOT / 'data'
IMAGE_DATA_DIR = DATA_DIR / 'images'/ 'images'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'non_generative_zero_shot_ablations'

MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

set_seed(seed)
if save:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print({'save': save, 'generate_submission': generate_submission, 'sanity_check': sanity_check, 'n': n})
print({'device': DEVICE, 'max_val_examples': max_val_examples, 'top_k': top_k, 'batch_size': batch_size})
print({'image_data_dir': str(IMAGE_DATA_DIR), 'out_dir': str(OUT_DIR)})


{'save': True, 'generate_submission': False, 'sanity_check': True, 'n': 512}
{'device': 'cuda', 'max_val_examples': 128, 'top_k': 5, 'batch_size': 64}
{'image_data_dir': '/content/drive/MyDrive/DL_Final_Project/data/images/images', 'out_dir': '/content/drive/MyDrive/DL_Final_Project/outputs/non_generative_zero_shot_ablations'}


## GPU / CUDA Check

This confirms whether inference runs on CUDA or CPU.

In [4]:
print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('cuda device count:', torch.cuda.device_count())
    print('current cuda device:', torch.cuda.current_device())
    print('cuda device name:', torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print('Running on CPU. Enable a GPU runtime in Colab for faster runs.')


torch version: 2.10.0+cu128
cuda available: True
cuda device count: 1
current cuda device: 0
cuda device name: NVIDIA A100-SXM4-40GB


## Load Data Splits

Sanity mode uses deterministic subsets for fast checks. Validation can be capped via `max_val_examples`.

In [5]:
val_df = load_split(DATA_DIR, 'val')
test_df = load_split(DATA_DIR, 'test')

val_df = apply_sanity_subset(val_df, sanity_check=sanity_check, n=n, seed=seed)
test_df = apply_sanity_subset(test_df, sanity_check=sanity_check, n=n, seed=seed)
val_df = cap_validation_rows(val_df, max_val_examples=max_val_examples)

print('val rows:', len(val_df))
print('test rows:', len(test_df))


val rows: 128
test rows: 512


## Load Processor + Model

No gradient updates are used in this notebook.

In [6]:
print('transformers version:', transformers.__version__)
if hasattr(transformers, 'AutoModelForVision2Seq'):
    ModelClass = transformers.AutoModelForVision2Seq
elif hasattr(transformers, 'AutoModelForImageTextToText'):
    ModelClass = transformers.AutoModelForImageTextToText
else:
    raise ImportError('Upgrade transformers: missing SmolVLM model class.')

processor = AutoProcessor.from_pretrained(MODEL_ID)
# Batched generation with decoder-only models should left-pad.
if hasattr(processor, 'tokenizer') and processor.tokenizer is not None:
    processor.tokenizer.padding_side = 'left'
    if processor.tokenizer.pad_token is None and processor.tokenizer.eos_token is not None:
        processor.tokenizer.pad_token = processor.tokenizer.eos_token

model = ModelClass.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()


transformers version: 5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Idefics3ForConditionalGeneration(
  (model): Idefics3Model(
    (vision_model): Idefics3VisionTransformer(
      (embeddings): Idefics3VisionEmbeddings(
        (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), padding=valid)
        (position_embedding): Embedding(1024, 768)
      )
      (encoder): Idefics3Encoder(
        (layers): ModuleList(
          (0-11): 12 x Idefics3EncoderLayer(
            (self_attn): Idefics3VisionAttention(
              (k_proj): Linear(in_features=768, out_features=768, bias=True)
              (v_proj): Linear(in_features=768, out_features=768, bias=True)
              (q_proj): Linear(in_features=768, out_features=768, bias=True)
              (out_proj): Linear(in_features=768, out_features=768, bias=True)
            )
            (layer_norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
            (mlp): Idefics3VisionMLP(
              (activation_fn): GELUTanh()
              (fc1): Linear(in_features=768, out

## Block-Tuning Revision (A-E)

This section mirrors the staged tuning structure used in the generative notebook.

Each block has:
1) config/search-space definition + config table display,
2) block execution,
3) block results table display.

In [7]:
# Shared non-generative block runner utilities.
NONGEN_BLOCK_KEYS = [
    'prompt_structure', 'context_mode', 'choice_format', 'choice_order', 'scoring_target', 'length_normalize',
    'image_size', 'enable_image_augmentation', 'brightness_jitter', 'slight_rotation_deg',
    'include_metadata_in_prompt', 'metadata_fields',
    'question_label', 'choices_label', 'answer_prefix_text', 'instruction_prefix',
]

NONGEN_RESULTS_BASE_COLS = [
    'ablation_id', 'percent_correct', 'accuracy', 'mean_confidence_margin',
    'prompt_structure', 'context_mode', 'choice_format', 'choice_order', 'scoring_target', 'length_normalize',
    'image_size', 'enable_image_augmentation', 'brightness_jitter', 'slight_rotation_deg',
    'include_metadata_in_prompt', 'metadata_fields',
    'question_label', 'choices_label', 'answer_prefix_text', 'instruction_prefix',
]

def run_nongen_block(block_name: str, block_prefix: str, fixed_values: dict, block_space: dict, selected_ids=None):
    all_cfgs = build_block_configs(
        ordered_keys=NONGEN_BLOCK_KEYS,
        fixed_values=fixed_values,
        block_space=block_space,
        ablation_prefix=block_prefix,
    )
    print(f'\n=== {block_name} ===')
    print('total configs:', len(all_cfgs))
    display(pd.DataFrame(all_cfgs))

    cfgs_to_run = select_ablation_configs(all_cfgs, selected_ablation_ids=selected_ids)
    print('configs to run now:', len(cfgs_to_run))

    all_results = []
    all_preds = []
    for cfg in cfgs_to_run:
        print(f"Running {cfg['ablation_id']} ...")
        pred_df = run_nongenerative_ablation(
            df=val_df,
            images_root=IMAGE_DATA_DIR,
            processor=processor,
            model=model,
            device=DEVICE,
            config=cfg,
            batch_size=batch_size,
        )
        all_preds.append(pred_df)
        all_results.append(summarize_nongenerative_predictions(pred_df, config=cfg))

    results = pd.DataFrame(all_results).sort_values('accuracy', ascending=False).reset_index(drop=True)
    preds = pd.concat(all_preds, ignore_index=True) if all_preds else pd.DataFrame()
    display_cols = [c for c in NONGEN_RESULTS_BASE_COLS if c in results.columns]
    return all_cfgs, results, preds, display_cols

# Baseline defaults for block tuning.
nongen_base_defaults = {
    'prompt_structure': 'question_first',
    'context_mode': 'hint_lecture',
    'choice_format': 'letter_dot',
    'choice_order': 'original',
    'scoring_target': 'letter',
    'length_normalize': True,
    'image_size': 384,
    'enable_image_augmentation': False,
    'brightness_jitter': 0.0,
    'slight_rotation_deg': 0.0,
    'include_metadata_in_prompt': True,
    'metadata_fields': ['subject', 'grade', 'topic'],
    'question_label': 'Question:',
    'choices_label': 'Choices:',
    'answer_prefix_text': 'Answer:',
    'instruction_prefix': '',
}

## Block A - Prompt/Context Backbone

Tune prompt layout and context inclusion while holding scoring/data controls fixed.

In [8]:
# Block A config space.
blockA_fixed = dict(nongen_base_defaults)
blockA_space = {
    'prompt_structure': ['explicit_instruction', 'question_first'],
    'context_mode': ['hint_only', 'lecture_only', 'hint_lecture'],
}
blockA_selected_ids = None
blockA_cfgs = build_block_configs(NONGEN_BLOCK_KEYS, blockA_fixed, blockA_space, 'nongenA')
print('Block A planned configs:', len(blockA_cfgs))
display(pd.DataFrame(blockA_cfgs))

Block A planned configs: 6


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,explicit_instruction,hint_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_001
1,explicit_instruction,lecture_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_002
2,explicit_instruction,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_003
3,question_first,hint_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_004
4,question_first,lecture_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_005
5,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_006


In [9]:
# Run Block A.
_, blockA_results_df, blockA_val_predictions_df, blockA_display_cols = run_nongen_block(
    block_name='Block A (Prompt/Context)',
    block_prefix='nongenA',
    fixed_values=blockA_fixed,
    block_space=blockA_space,
    selected_ids=blockA_selected_ids,
)


=== Block A (Prompt/Context) ===
total configs: 6


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,explicit_instruction,hint_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_001
1,explicit_instruction,lecture_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_002
2,explicit_instruction,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_003
3,question_first,hint_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_004
4,question_first,lecture_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_005
5,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenA_006


configs to run now: 6
Running nongenA_001 ...


NonGen nongenA_001:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenA_002 ...


...

NonGen nongenA_004:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenA_005 ...


NonGen nongenA_005:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenA_006 ...


NonGen nongenA_006:   0%|          | 0/2 [00:00<?, ?it/s]

In [10]:
# Display Block A results table.
display(blockA_results_df[blockA_display_cols])
print('Top Block A configs:')
#print(json.dumps(top_configs(blockA_results_df, k=top_k), indent=2))

,ablation_id,percent_correct,accuracy,mean_confidence_margin,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,nongenA_001,30.46875,0.304688,2.423096,explicit_instruction,hint_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
1,nongenA_002,30.46875,0.304688,2.520752,explicit_instruction,lecture_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
2,nongenA_003,30.46875,0.304688,2.235840,explicit_instruction,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
3,nongenA_004,30.46875,0.304688,3.505981,question_first,hint_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
4,nongenA_005,30.46875,0.304688,3.799072,question_first,lecture_only,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
5,nongenA_006,30.46875,0.304688,3.381836,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,


Top Block A configs:


## Block B - Image/Data Input Ablations

Tune image resizing and metadata-in-prompt controls.

In [13]:
import gc
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

In [14]:
# Block B config space.
blockB_fixed = dict(nongen_base_defaults)
blockB_space = {
    'image_size': [None, 336, 384],
    'include_metadata_in_prompt': [False, True],
    'metadata_fields': [['subject', 'grade', 'topic'], ['subject', 'topic']],
}
blockB_selected_ids = None
blockB_cfgs = build_block_configs(NONGEN_BLOCK_KEYS, blockB_fixed, blockB_space, 'nongenB')
print('Block B planned configs:', len(blockB_cfgs))
display(pd.DataFrame(blockB_cfgs))

Block B planned configs: 12


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_001
1,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_002
2,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_003
3,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_004
4,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_005
5,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_006
6,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_007
7,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_008
8,question_first,hint_lecture,letter_dot,original,letter,True,384.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_009
9,question_first,hint_lecture,letter_dot,original,letter,True,384.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_010


In [15]:
# Run Block B.
_, blockB_results_df, blockB_val_predictions_df, blockB_display_cols = run_nongen_block(
    block_name='Block B (Image/Data Input)',
    block_prefix='nongenB',
    fixed_values=blockB_fixed,
    block_space=blockB_space,
    selected_ids=blockB_selected_ids,
)


=== Block B (Image/Data Input) ===
total configs: 12


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_001
1,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_002
2,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_003
3,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_004
4,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_005
5,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_006
6,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_007
7,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_008
8,question_first,hint_lecture,letter_dot,original,letter,True,384.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenB_009
9,question_first,hint_lecture,letter_dot,original,letter,True,384.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,,nongenB_010


configs to run now: 12
Running nongenB_001 ...


NonGen nongenB_001:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenB_002 ...


...

NonGen nongenB_010:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenB_011 ...


NonGen nongenB_011:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenB_012 ...


NonGen nongenB_012:   0%|          | 0/2 [00:00<?, ?it/s]

In [17]:
# Display Block B results table.
display(blockB_results_df[blockB_display_cols])
print('Top Block B configs:')
#print(json.dumps(top_configs(blockB_results_df, k=top_k), indent=2))

,ablation_id,percent_correct,accuracy,mean_confidence_margin,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,nongenB_001,30.46875,0.304688,2.577759,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,
1,nongenB_002,30.46875,0.304688,2.577759,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,
2,nongenB_003,30.46875,0.304688,3.394653,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
3,nongenB_004,30.46875,0.304688,2.985596,question_first,hint_lecture,letter_dot,original,letter,True,NaN,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,Answer:,
4,nongenB_005,30.46875,0.304688,2.629028,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,
5,nongenB_006,30.46875,0.304688,2.629028,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,
6,nongenB_007,30.46875,0.304688,3.381836,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
7,nongenB_008,30.46875,0.304688,2.989746,question_first,hint_lecture,letter_dot,original,letter,True,336.0,False,0.0,0.0,True,"[subject, topic]",Question:,Choices:,Answer:,
8,nongenB_009,30.46875,0.304688,2.629028,question_first,hint_lecture,letter_dot,original,letter,True,384.0,False,0.0,0.0,False,"[subject, grade, topic]",Question:,Choices:,Answer:,
9,nongenB_010,30.46875,0.304688,2.629028,question_first,hint_lecture,letter_dot,original,letter,True,384.0,False,0.0,0.0,False,"[subject, topic]",Question:,Choices:,Answer:,


Top Block B configs:


## Block C - Prompt Phrasing Overrides

Tune labels and answer-prefix phrasing in the prompt contract.

In [18]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

In [19]:
# Block C config space.
blockC_fixed = dict(nongen_base_defaults)
blockC_space = {
    #'question_label': ['Question:', 'Prompt:'],
    #'choices_label': ['Choices:', 'Options:'],
    'answer_prefix_text': ['Answer:', 'The correct answer is:'],
    'instruction_prefix': ['', 'Use the image and context carefully.'],
}
blockC_selected_ids = None
blockC_cfgs = build_block_configs(NONGEN_BLOCK_KEYS, blockC_fixed, blockC_space, 'nongenC')
print('Block C planned configs:', len(blockC_cfgs))
display(pd.DataFrame(blockC_cfgs))

Block C planned configs: 4


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenC_001
1,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,Use the image and context carefully.,nongenC_002
2,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,nongenC_003
3,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,Use the image and context carefully.,nongenC_004


In [20]:
# Run Block C.
_, blockC_results_df, blockC_val_predictions_df, blockC_display_cols = run_nongen_block(
    block_name='Block C (Prompt Phrasing)',
    block_prefix='nongenC',
    fixed_values=blockC_fixed,
    block_space=blockC_space,
    selected_ids=blockC_selected_ids,
)


=== Block C (Prompt Phrasing) ===
total configs: 4


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenC_001
1,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,Use the image and context carefully.,nongenC_002
2,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,,nongenC_003
3,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,Use the image and context carefully.,nongenC_004


configs to run now: 4
Running nongenC_001 ...


NonGen nongenC_001:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenC_002 ...


NonGen nongenC_002:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenC_003 ...


NonGen nongenC_003:   0%|          | 0/2 [00:00<?, ?it/s]

Running nongenC_004 ...


NonGen nongenC_004:   0%|          | 0/2 [00:00<?, ?it/s]

In [21]:
# Display Block C results table.
display(blockC_results_df[blockC_display_cols])
print('Top Block C configs:')
#print(json.dumps(top_configs(blockC_results_df, k=top_k), indent=2))

,ablation_id,percent_correct,accuracy,mean_confidence_margin,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix
0,nongenC_001,30.46875,0.304688,3.381836,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,
1,nongenC_002,30.46875,0.304688,3.381836,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,Use the image and context carefully.
2,nongenC_003,30.46875,0.304688,4.926697,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,
3,nongenC_004,30.46875,0.304688,4.926697,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,The correct answer is:,Use the image and context carefully.


Top Block C configs:


## Block D - Non-Generative Objective-Specific Controls

Tune scoring objective, normalization, and option-format/order controls.

In [22]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

In [23]:
# Block D config space.
blockD_fixed = dict(nongen_base_defaults)
blockD_space = {
    #'scoring_target': ['letter', 'answer_prefix'],
    'length_normalize': [False, True],
    'choice_format': ['letter_dot', 'letter_paren', 'option_letter'],
    #'choice_order': ['original', 'reverse'],
}
blockD_selected_ids = None
blockD_cfgs = build_block_configs(NONGEN_BLOCK_KEYS, blockD_fixed, blockD_space, 'nongenD')
print('Block D planned configs:', len(blockD_cfgs))
display(pd.DataFrame(blockD_cfgs))

Block D planned configs: 6


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,original,letter,False,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_001
1,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_002
2,question_first,hint_lecture,letter_paren,original,letter,False,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_003
3,question_first,hint_lecture,letter_paren,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_004
4,question_first,hint_lecture,option_letter,original,letter,False,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_005
5,question_first,hint_lecture,option_letter,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_006


In [24]:
# Run Block D.
_, blockD_results_df, blockD_val_predictions_df, blockD_display_cols = run_nongen_block(
    block_name='Block D (Non-Generative Objective Controls)',
    block_prefix='nongenD',
    fixed_values=blockD_fixed,
    block_space=blockD_space,
    selected_ids=blockD_selected_ids,
)


=== Block D (Non-Generative Objective Controls) ===
total configs: 6


,prompt_structure,context_mode,choice_format,choice_order,scoring_target,length_normalize,image_size,enable_image_augmentation,brightness_jitter,slight_rotation_deg,include_metadata_in_prompt,metadata_fields,question_label,choices_label,answer_prefix_text,instruction_prefix,ablation_id
0,question_first,hint_lecture,letter_dot,original,letter,False,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_001
1,question_first,hint_lecture,letter_dot,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_002
2,question_first,hint_lecture,letter_paren,original,letter,False,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_003
3,question_first,hint_lecture,letter_paren,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_004
4,question_first,hint_lecture,option_letter,original,letter,False,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_005
5,question_first,hint_lecture,option_letter,original,letter,True,384,False,0.0,0.0,True,"[subject, grade, topic]",Question:,Choices:,Answer:,,nongenD_006


configs to run now: 6
Running nongenD_001 ...


NonGen nongenD_001:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Display Block D results table.
display(blockD_results_df[blockD_display_cols])
print('Top Block D configs:')
#print(json.dumps(top_configs(blockD_results_df, k=top_k), indent=2))

## Block E - Augmentation Stress Test

Tune augmentation controls for robustness-oriented ablations.

In [16]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # optional; can help in long notebook runs

In [ ]:
# Block E config space.
blockE_fixed = dict(nongen_base_defaults)
blockE_space = {
    'brightness_jitter': [0.0, 0.05, 0.10],
    'slight_rotation_deg': [0.0, 2.5, 5.0],
}
blockE_selected_ids = None
blockE_cfgs = build_block_configs(NONGEN_BLOCK_KEYS, blockE_fixed, blockE_space, 'nongenE')
print('Block E planned configs:', len(blockE_cfgs))
display(pd.DataFrame(blockE_cfgs))

In [ ]:
# Run Block E.
_, blockE_results_df, blockE_val_predictions_df, blockE_display_cols = run_nongen_block(
    block_name='Block E (Augmentation Stress Test)',
    block_prefix='nongenE',
    fixed_values=blockE_fixed,
    block_space=blockE_space,
    selected_ids=blockE_selected_ids,
)

In [ ]:
# Display Block E results table.
display(blockE_results_df[blockE_display_cols])
print('Top Block E configs:')
#print(json.dumps(top_configs(blockE_results_df, k=top_k), indent=2))

# Final outputs are sourced from Block E by default.
results_df = blockE_results_df.copy()
val_predictions_df = blockE_val_predictions_df.copy()
top_cfgs = top_configs(results_df, k=top_k)

## Save Revised Block Outputs

This save section persists artifacts for the final block executed in the revised A-E pipeline.

In [ ]:
# Save only per-block results DataFrames (A-E).
maybe_save_dataframe(blockA_results_df, OUT_DIR / 'blockA_results.csv', save=save)
maybe_save_dataframe(blockB_results_df, OUT_DIR / 'blockB_results.csv', save=save)
maybe_save_dataframe(blockC_results_df, OUT_DIR / 'blockC_results.csv', save=save)
maybe_save_dataframe(blockD_results_df, OUT_DIR / 'blockD_results.csv', save=save)
maybe_save_dataframe(blockE_results_df, OUT_DIR / 'blockE_results.csv', save=save)
print('Saved block result DataFrames only (A-E).')